# Cocogoat 2026 - Task 4: prompt-conditional 1024x1024 GAN (~10.5 h)

FFHQ-1024 (70k faces) downloaded from the Hugging Face mirror on Colab. Base GAN at 256x256 conditioned on CLIP ViT-L/14 embeddings (language-free LAFITE training: image embeddings + noise while training, text prompts at inference), with a pixel critic and a projected critic on frozen DINOv2-S features (~7 h). Then a x4 super-resolution GAN trained from scratch on real 1024 crops (~2.5 h). Final evaluation at 1024: KID/FID, patch-KID on native-resolution crops, precision/recall, CLIP score for text prompts. Outputs: `MyDrive/Cocogoat/runs/task4/`. Needs ~130 GB of the Colab disk (dataset + decoded arrays).

**How to run:** `Runtime -> Change runtime type -> G4 (RTX PRO 6000)`, set the budgets in the settings cell (5),
then `Runtime -> Run all`.
Keep this tab open (Colab Pro stops idle/closed sessions). Everything is written to
`MyDrive/Cocogoat/runs/...`; if Colab disconnects, just *Run all* again - training resumes from the last checkpoint.
You do not need to inspect results: metrics, sample grids and logs are saved to Drive for analysis.

In [ ]:
# 1) Mount Google Drive (outputs + datasets live there)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Get / update the code (public repo, branch claude/cocogoat-revival-glfnmz)
import os, subprocess
REPO, BRANCH = "https://github.com/Martyn0324/Cocogoat.git", "claude/cocogoat-revival-glfnmz"
if not os.path.exists('/content/Cocogoat/.git'):
    subprocess.run(['git', 'clone', '-b', BRANCH, '--depth', '1', REPO, '/content/Cocogoat'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Cocogoat', 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Cocogoat', 'reset', '--hard', 'FETCH_HEAD'], check=True)
print(subprocess.run(['git', '-C', '/content/Cocogoat', 'log', '-1', '--oneline'], capture_output=True, text=True).stdout)
os.chdir('/content/Cocogoat')

In [ ]:
# 3) Extra packages (torch/torchvision/transformers come with Colab)
!pip -q install -r /content/Cocogoat/requirements-colab.txt
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
# Colab Secrets are only readable from the notebook kernel: export HF_TOKEN for the training subprocesses
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN exported')
except Exception as exc:
    print('No HF_TOKEN secret (only needed for gated Hugging Face datasets):', type(exc).__name__)
os.environ['PYTHONUNBUFFERED'] = '1'

In [ ]:
# 4) GPU smoke test (tiny models on synthetic shards, a few steps, always from scratch)
!python -m cocogoat.tasks.task4_hires --smoke --run_dir /content/drive/MyDrive/Cocogoat/runs/task4_smoke --force

In [ ]:
# 5) Settings - edit freely. Training time budgets in hours (data/evaluation time not included).
#    MAX_SHARDS: FFHQ shards of ~1,000 faces each; 0 = all 70 (~64 GB download, ~20-30 min with decoding).
#    Quick test: e.g. MAX_SHARDS = 5 with small budgets, in a separate folder (e.g. '.../runs/task4_test').
#    Raising BASE_HOURS / SR_HOURS later on the same RUN_DIR continues training from the checkpoints.
MAX_SHARDS, BASE_HOURS, SR_HOURS = 0, 7.0, 2.5
RUN_DIR = '/content/drive/MyDrive/Cocogoat/runs/task4'

In [ ]:
# 6) Data (download + decode + CLIP embeddings), base GAN, SR, 1024 evaluation. Everything resumes: if Colab
#    disconnects, Run all again (the data step re-downloads on a fresh VM).
!python -m cocogoat.tasks.task4_hires --max_shards {MAX_SHARDS} --base_hours {BASE_HOURS} --sr_hours {SR_HOURS} --run_dir {RUN_DIR}

In [ ]:
# 7) Results
from IPython.display import Image, Markdown, display
run = RUN_DIR
display(Markdown(open(f'{run}/eval/report.md').read()))
display(Image(f'{run}/eval/samples/prompts_grid.png'))
display(Image(f'{run}/sr/samples/sr_final.png'))